# 언론사 전처리본 분석 토큰화 (Kiwi)

전처리본 `전처리_본문_언론사_{기간}.csv`를 읽어 분석용 토큰(`tokens`)을 만드는 노트북 — LDA·빈도·워드클라우드의 입력

- 구조정제(앞 노트북)와 분리된 2단계 — 가독 본문 `body_cleaned`는 그대로 두고, 여기서 토큰만 따로 생성
- 형태소 분석기는 **Kiwi**(신조어·고유명사·오타교정에 강함)
- `hangeul_only`(한글만 남김)는 안 씀 — `AI`·`HMM`·`美`·`韓` 같은 의제어가 사라지므로, **품사 기반**으로 명사(NNG)·고유명사(NNP)·영문(SL)·한자(SH)만 남김
- 불용어(내장 + 뉴스 상투어)로 `기자`·`연합뉴스` 등 제거, `keep_terms`로 핵심 의제어 보존

## 토큰화 방법

1. 입력은 전처리본의 `text`(= 정제 제목 + 정제 본문)
2. 가벼운 전처리 — HTML 엔티티 복원(`html.unescape`), 이모지 제거
3. Kiwi 형태소 분석 후 **명사류만 선별** — 품사 NNG/NNP/SL/SH, 길이 2 이상, 불용어 제외
4. `keep_terms`(의제 핵심어)는 사용자 사전 등록 + 무조건 보존
5. 산출 — `tokens`(공백으로 이어붙인 토큰), `n_tokens`(토큰 수) → `분석토큰_언론사_{기간}.csv`

불용어·keep_terms는 1차 LDA 결과를 보고 계속 보강 (상단 목록 수정 또는 `resources/` 파일로 관리)

In [ ]:
# Colab에서 실행할 때만 아래 3줄 주석 해제 — 로컬/WSL에서는 그대로 두기
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os, re, html, unicodedata
import pandas as pd

# Colab 기본 경로 — Drive 미마운트면 except로 빠져 로컬/WSL 경로 사용
try:
    PROJECT_DIR = Path('/content/drive/MyDrive/Text-data-Analysis_26-Spring')
    if not PROJECT_DIR.exists():
        raise FileNotFoundError
except Exception:
    PROJECT_DIR = Path('/home/carol/Text-data-Analysis_26-Spring')
os.chdir(PROJECT_DIR)
DATA_DIR = PROJECT_DIR / 'data' / 'news'

# 입력 전처리본 찾기 — 한글 파일명 NFC 정규화 후 매칭
def normalize_name(p): return unicodedata.normalize('NFC', p.name)
cands = sorted(p for p in DATA_DIR.iterdir()
               if p.is_file() and re.match(r'^전처리_본문_언론사_\d{6}_\d{6}\.csv$', normalize_name(p)))
if not cands:
    raise FileNotFoundError(f'전처리본을 찾지 못함: {DATA_DIR}')
INPUT_PATH = cands[-1]
print('입력 전처리본:', INPUT_PATH.name)
df = pd.read_csv(INPUT_PATH, encoding='utf-8-sig')
print('행수:', len(df), '/ 컬럼:', list(df.columns))

In [ ]:
# --- Kiwi와 불용어·핵심어 세팅 ---
from kiwipiepy import Kiwi
from kiwipiepy.utils import Stopwords

# typos 인자는 kiwipiepy 버전에 따라 init/analyze 위치가 달라 호환 위해 try
try:
    kiwi = Kiwi(typos='basic')   # 기본 오타 교정
except TypeError:
    kiwi = Kiwi()
stopwords = Stopwords()          # 내장 불용어

# 뉴스 상투어 불용어 시드 — 1차 LDA 결과 보고 계속 보강
NEWS_STOP = {
    '기자','특파원','논설위원','뉴스','보도','취재','기사','오늘','이날','어제','내일',
    '관련','위해','통해','지난','당시','이번','최근','대해','경우','정도','가운데','이후','현재','상황','모습',
    '사진','자료','제공','출처','무단','전재','재배포','배포','금지','연합뉴스','뉴스원','뉴시스',
    '뉴스1','연합뉴스TV','제보','구독','한겨레','조선일보','한국경제','매일경제','방송','앵커','리포트','종합','속보','단독','촬영','영상','편집',
    # 보강 — Kiwi가 한 토큰으로 만드는 합성 제작진어·매체/캡션 노이즈 (제작/구성/연출/디자인 같은 이중용도 낱말은 본문 과제거 우려로 제외)
    '영상편집','영상취재','촬영기자','자막그래픽','디지털뉴스부','뉴미디어부',
    '갈무리','일러스트','에디터','이데일리','자막뉴스','현장영상','취재파일','포토','서울앤','웨더아이',
}
for w in NEWS_STOP:
    stopwords.add((w, 'NNG'))
stopwords_set = set(NEWS_STOP)
try:
    stopwords_set |= {w for w, _ in stopwords.stopwords}
except Exception:
    pass

# keep_terms — 분석에서 보존할 핵심 의제어(영문 약어·한자·고유표현). resources/keep_terms.txt 있으면 합침
KEEP_TERMS = {
    'AI','HMM','ETF','MOU','LNG','CEO','GDP','OLED','UAE','JWST','SNS','TV','IT','EU','UN',
    '美','韓','中','日','北','與','野','靑','尹','호르무즈','반도체','금리','환율','관세',
    '장동혁','박민식','정원오','오세훈','하정우',   # 6·3 지방선거 주요 인물(분해 방지) — 주간 갱신
}
kt_file = PROJECT_DIR / 'resources' / 'keep_terms.txt'
if kt_file.exists():
    KEEP_TERMS |= {ln.strip() for ln in kt_file.read_text(encoding='utf-8').splitlines() if ln.strip()}
keep_terms_set = set(KEEP_TERMS)
stopwords_set -= keep_terms_set                 # 핵심어가 불용어에 있으면 빼기
for term in keep_terms_set:
    kiwi.add_user_word(term, tag='NNG')         # 핵심어를 명사로 사용자 사전 등록 → 분해 방지
print('불용어:', len(stopwords_set), '/ keep_terms:', len(keep_terms_set))

In [ ]:
# --- 전처리·토큰화 함수 ---
import emoji

# 명사류만 — 일반명사·고유명사·외국어(영문)·한자. 숫자(SN)·조사·어미·기호는 제외
KEEP_POS = {'NNG', 'NNP', 'SL', 'SH'}

def pre_clean(text):
    t = html.unescape(str(text))                # &amp; 같은 엔티티 복원
    t = emoji.replace_emoji(t, replace=' ')     # 이모지 제거
    return t

def tokenize_doc(text):
    toks = []
    for tok in kiwi.tokenize(pre_clean(text)):
        f = tok.form
        if f in keep_terms_set:                                  # 핵심 의제어는 무조건 보존
            keep = True
        elif tok.tag in KEEP_POS and len(f) >= 2 and f not in stopwords_set:
            keep = True
        else:
            keep = False
        if not keep:
            continue
        f = f.replace(' ', '_').strip('·:/.,“”\'"')   # 다어절 NNP 내부공백->_ (공백조인 저장 분해 방지), 양끝 기호 정리
        if f:
            toks.append(f)
    return toks

# 동작 확인
sample = "국민의힘 장동혁 대표가 美·韓 협력과 AI·반도체 정책을 강조했다 김재훈 기자 연합뉴스"
print('샘플 토큰:', tokenize_doc(sample))

In [ ]:
# --- 전체 토큰화 + 저장 ---
try:
    from tqdm.auto import tqdm
    tqdm.pandas(desc='토큰화')
    tok_lists = df['text'].progress_map(tokenize_doc)
except Exception:
    tok_lists = df['text'].map(tokenize_doc)

df['tokens'] = tok_lists.map(lambda xs: ' '.join(xs))   # 공백으로 이어붙여 저장(재로딩 시 split 한 줄)
df['n_tokens'] = tok_lists.map(len)

out_cols = ['article_id', 'press', 'media_group', 'date', 'article_category', 'title_cleaned', 'n_tokens', 'tokens']
period = re.search(r'(\d{6}_\d{6})', normalize_name(INPUT_PATH)).group(1)
OUT_PATH = DATA_DIR / f'분석토큰_언론사_{period}.csv'
df[out_cols].to_csv(OUT_PATH, index=False, encoding='utf-8-sig')
print('저장 완료:', OUT_PATH)
print('n_tokens 분포:'); print(df['n_tokens'].describe())

In [ ]:
# --- 검증·현황 ---
from collections import Counter
print('=== 매체그룹별 평균 토큰 수 ===')
print(df.groupby('media_group')['n_tokens'].mean().round(1))
print('\n=== 전체 상위 빈도 단어 30 (불용어·상투어가 안 보여야 정상) ===')
allc = Counter()
for s in df['tokens']:
    allc.update(s.split())
for w, c in allc.most_common(30):
    print(f'  {w} {c}')
print('\n=== 토큰 샘플 3건 ===')
for s in df['tokens'].head(3):
    print(' ', s[:160])